# Session 1 — inspect one night before modeling

We want to understand what each sensor measures, how timestamps align, and where data is missing. No features are imputed and no model is trained here.

Run the setup/download commands in the README first. This notebook imports the installed package; it works from the repository root or the `notebooks` directory. In Colab, clone the repository, install uv, and run the same commands using the checkout path. Use the uv-managed Jupyter kernel locally; see README for Colab's separate kernel environment.


In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

from sleep_classifier.data import audit_subject, load_config, load_sensor, sensor_path

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "configs/explore.yaml").exists())
config = load_config(ROOT / "configs/explore.yaml")
report = audit_subject(config["data_dir"], config["subject_id"])
pd.DataFrame(report["sensors"]).T

## What does the clock mean?

Times are seconds relative to the start of polysomnography. Negative values are history before that recording. Label timestamps are spaced 30 seconds apart in this sample; that alone does not prove which boundary the label denotes.

The audit preserves original row order so timestamp resets and duplicates remain visible. Plots below sort a temporary copy for display only; this is not a cleaning policy.


In [ ]:
frames = {
    sensor: load_sensor(sensor_path(config["data_dir"], config["subject_id"], sensor), sensor)
    for sensor in report["sensors"]
}
hr = frames["heart_rate"]
backward = hr.index[hr.time.diff() < 0]
for index in backward:
    display(hr.iloc[max(0, index - 2) : index + 3])

## Signals during the labeled night

The motion plot uses a one-second display average of acceleration magnitude in g, including gravity. It is not a model feature or a movement score. Heart-rate points remain separate: connecting them across gaps would suggest measurements we do not have.


In [ ]:
labels = frames["labels"].sort_values("time")
end = labels.time.max() + 30

fig, axes = plt.subplots(3, 1, figsize=(12, 8), sharex=True)

night_hr = hr[hr.time.between(0, end, inclusive="left")]

axes[0].scatter(night_hr.time / 3600, night_hr.heart_rate, s=3)
axes[0].set_ylabel("Heart rate (bpm)")

motion = frames["motion"]
night_motion = motion[motion.time.between(0, end, inclusive="left")].copy()
night_motion["magnitude"] = (night_motion[["x", "y", "z"]] ** 2).sum(axis=1) ** 0.5

seconds = night_motion.groupby(night_motion.time.astype(int)).magnitude.mean()
seconds = seconds.reindex(range(int(end)))  # Leave missing seconds as gaps in the plot

axes[1].plot(seconds.index / 3600, seconds.values, linewidth=0.5)
axes[1].set_ylabel("Acceleration (g)")
axes[2].step(labels.time / 3600, labels.stage, where="post")
axes[2].set_yticks([0, 1, 2, 3, 5], ["Wake", "N1", "N2", "N3", "REM"])
axes[2].set_xlabel("Hours from PSG start")

fig.suptitle(f"Participant {config['subject_id']} — observed signals")

plt.tight_layout()
plt.show()

## Label balance and historical step coverage

Counts describe this participant only. Step counts refer to bins ending at the next timestamp; the final bin has no documented endpoint in the file. Empty coverage is not evidence of zero activity.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 3))

labels.stage.value_counts().sort_index().rename(
    index={0: "Wake", 1: "N1", 2: "N2", 3: "N3", 5: "REM"}
).plot.bar(ax=axes[0])
axes[0].set_ylabel("30-second epochs")

steps = frames["steps"].sort_values("time")
axes[1].scatter(steps.time / 86400, steps.steps, s=5)
axes[1].axvspan(-1, 0, color="orange", alpha=0.2, label="24h before PSG")
axes[1].set_xlabel("Days relative to PSG start")
axes[1].set_ylabel("Steps per recorded bin")
axes[1].legend()

plt.tight_layout()
plt.show()

## Decisions to make together

- Are repeated heart-rate timestamps exact duplicate records or conflicting readings?
- How much of each labeled epoch has sensor coverage? What gap threshold should invalidate a feature?
- Given incomplete step history, should the first baseline omit steps or explicitly model coverage?
- Confirm label boundary semantics before implementing causal epoch aggregation.

Next milestone: agree on these policies, audit the cohort, and implement tested preparation. The sample inspected here is exploratory development data and should be excluded from the final untouched test split.
